# SFT: Teach Qwen3.5-0.8B DSPy Structured Output Format

Fine-tune Qwen3.5-0.8B to produce DSPy ChatAdapter's `[[ ## field ## ]]` format.

Based on [Unsloth Qwen3.5 (0.8B) Vision notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(0_8B)_Vision.ipynb).

**Training data:** `format_train.jsonl` — ~4K examples across 5 modules, generated using DSPy's own ChatAdapter to guarantee format match.

**Note:** Qwen3.5 is a VLM architecture — we use `FastVisionModel` even though our task is text-only.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-0.8B",
    load_in_4bit=False,
    use_gradient_checkpointing="unsloth",
)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,      # text-only task, skip vision encoder
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

## Load Training Data

Upload `format_train.jsonl` to Colab first (Files panel or `from google.colab import files`).

Each line: `{"module": "...", "messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}`

In [ ]:
import json
from collections import Counter

# Upload the file first:
# from google.colab import files
# uploaded = files.upload()  # upload format_train.jsonl

DATA_PATH = "format_train.jsonl"

with open(DATA_PATH) as f:
    raw = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(raw)} examples")
print("Module distribution:", Counter(r["module"] for r in raw))

In [ ]:
# Convert to the format FastVisionModel expects:
# list of dicts with "messages" key, where content is list of typed blocks
converted_dataset = []
for r in raw:
    messages = []
    for msg in r["messages"]:
        messages.append({
            "role": msg["role"],
            "content": [{"type": "text", "text": msg["content"]}],
        })
    converted_dataset.append({"messages": messages})

print(f"Converted {len(converted_dataset)} examples")
print(f"Sample roles: {[m['role'] for m in converted_dataset[0]['messages']]}")
print(f"Sample assistant content (first 200 chars): {converted_dataset[0]['messages'][-1]['content'][0]['text'][:200]}")

In [ ]:
# 90/10 train/val split
import random
random.seed(42)
indices = list(range(len(converted_dataset)))
random.shuffle(indices)
split_idx = int(len(indices) * 0.9)
train_data = [converted_dataset[i] for i in indices[:split_idx]]
val_data = [converted_dataset[i] for i in indices[split_idx:]]
print(f"Train: {len(train_data)}, Val: {len(val_data)}")

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_data,
    eval_dataset=val_data,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=16384,
    ),
)

In [ ]:
# Check GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# Memory stats after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']:.0f} seconds used for training.")
print(f"{trainer_stats.metrics['train_runtime']/60:.1f} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")

In [ ]:
# Eval loss
eval_results = trainer.evaluate()
print(f"Eval loss: {eval_results['eval_loss']:.4f}")

## Test Inference

Verify the fine-tuned model produces `[[ ## field ## ]]` format correctly.

In [ ]:
FastVisionModel.for_inference(model)

# Test intent_decider
test_messages = [
    {"role": "user", "content": [
        {"type": "text", "text": """Your input fields are:\n1. `context` (str): Form fields and current state\n2. `user_message` (str): Current user message\nYour output fields are:\n1. `intent` (Literal['gather', 'converse', 'clarify', 'close', 'review']): The assistant's intent for this turn\n\n[[ ## context ## ]]\nFilled fields: (none)\nFields filled: 0\n\n[[ ## user_message ## ]]\nHi, I want to apply for the CS program.\n\nRespond with the corresponding output fields, starting with the field `[[ ## intent ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`."""}
    ]}
]

input_text = tokenizer.apply_chat_template(test_messages, add_generation_prompt=True)
inputs = tokenizer(
    None,  # no image
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
print("=== Intent Decider ===")
_ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=128,
                   use_cache=True, temperature=0.1)

In [ ]:
# Test data_extractor
test_messages_extract = [
    {"role": "user", "content": [
        {"type": "text", "text": """Your input fields are:\n1. `context` (str): Form fields and current state\n2. `user_message` (str): User message containing form data\nYour output fields are:\n1. `field_ids` (list[str]): List of field_ids to set, e.g. ['full_name', 'email']\n2. `field_values` (list[str]): Corresponding values, e.g. ['Jane Smith', 'jane@email.com']\n\n[[ ## context ## ]]\nFilled fields: {"program": "cs"}\nFields filled: 1\n\n[[ ## user_message ## ]]\nMy name is Jane Smith and my email is jane@example.com\n\nRespond with the corresponding output fields, starting with the field `[[ ## field_ids ## ]]` (must be formatted as a valid Python list[str]), then `[[ ## field_values ## ]]` (must be formatted as a valid Python list[str]), and then ending with the marker for `[[ ## completed ## ]]`."""}
    ]}
]

input_text = tokenizer.apply_chat_template(test_messages_extract, add_generation_prompt=True)
inputs = tokenizer(
    None,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

print("=== Data Extractor ===")
_ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=128,
                   use_cache=True, temperature=0.1)

## Save Model

In [ ]:
# Save LoRA adapter
model.save_pretrained("qwen35-08b-dspy-format-lora")
tokenizer.save_pretrained("qwen35-08b-dspy-format-lora")
print("Saved LoRA adapter")

In [ ]:
# Save merged model as GGUF Q8_0 for local inference with mlx/llama.cpp
model.save_pretrained_gguf(
    "qwen35-08b-dspy-format-gguf",
    tokenizer,
    quantization_method="q8_0",
)
print("Saved GGUF Q8_0")

In [ ]:
# Download the GGUF file
from google.colab import files
import glob
gguf_files = glob.glob("qwen35-08b-dspy-format-gguf/*.gguf")
for f in gguf_files:
    print(f"Downloading {f}...")
    files.download(f)